# RAISE PDF Parsing

Owner: Person B  
Repository: `semanticClimate/RAISE`  
Branch: `Harsh-semantification`  
Module folder: `RAISE_PDF_Parsing`

Task: convert a raw university annual-report PDF into `report_blocks.json` (Stage 1: layout-aware parsing), then filter it into `target_blocks.json` (Stage 2: token-budget filtering) -- the file the Stage 3 owner's `RAISE_HTML5_Semantification` notebook expects as input.

Upload a `.pdf` annual report.

## Upload the Raw Annual Report PDF

This cell opens the Colab file picker, records the uploaded PDF filename in `input_pdf_filename`, and confirms the selected input. It is required because Stage 1 reads the raw report from that filename.

In [ ]:
from google.colab import files

uploaded = files.upload()
input_pdf_filename = next(iter(uploaded))
print(f"Uploaded PDF file: {input_pdf_filename}")

## Upload an Optional Content Schema

Set `UPLOAD_CUSTOM_SCHEMA` to `True` to upload a JSON Schema. It is evaluated against each Stage 2 block: matching blocks are kept, while failures are removed and written to `schema_filtering_report.json`. Leave it `False` to use only the built-in filters.

In [ ]:
import json

UPLOAD_CUSTOM_SCHEMA = False  # @param {type:"boolean"}
custom_schema = None
if UPLOAD_CUSTOM_SCHEMA:
    print("Choose one JSON Schema file.")
    schema_upload = files.upload()
    if len(schema_upload) != 1:
        raise ValueError("Upload exactly one JSON Schema file.")
    schema_filename, schema_bytes = next(iter(schema_upload.items()))
    if not schema_filename.lower().endswith(".json"):
        raise ValueError("The custom schema must be a .json file.")
    custom_schema = json.loads(schema_bytes.decode("utf-8"))
    print(f"Loaded custom schema: {schema_filename}")
else:
    print("Custom schema filtering disabled.")

## Install Runtime Dependencies

This cell installs the Python libraries required for layout-aware PDF parsing, table extraction, data models, and command-line support. It is included so a fresh Colab runtime has the same required dependencies before the parsing code runs.

In [ ]:
!pip install -q pymupdf pdfplumber pydantic typer rich jsonschema

## Install the RAISE Module from GitHub

This cell installs `RAISE_PDF_Parsing` directly from the `Harsh-semantification` branch of the `semanticClimate/RAISE` repository. It is required so the Colab runtime uses the committed GitHub package and stops immediately if that package cannot be installed.

In [ ]:
# Install from the exact RAISE monorepo branch and module folder when opened from GitHub Colab.

import subprocess
import sys

GITHUB_PACKAGE = (
    "git+https://github.com/semanticClimate/RAISE.git"
    "@Harsh-semantification#subdirectory=RAISE_PDF_Parsing"
)

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", GITHUB_PACKAGE])
print("Installed raise_pdf_parsing from GitHub.")

## Import the Installed Repository Engine

This cell imports the production parsing + filtering API from the `raise_pdf_parsing` package installed by the GitHub cell. It loads the repository functions for Stage 1 (PDF -> blocks) and Stage 2 (blocks -> filtered, research-only blocks).

In [ ]:
import json
from pathlib import Path

import raise_pdf_parsing
from raise_pdf_parsing.main import run_pipeline
from raise_pdf_parsing.models import TargetBlocksDocument
from raise_pdf_parsing.schema_filter import filter_target_by_schema, validate_custom_schema

if custom_schema is not None:
    validate_custom_schema(custom_schema)
    print("Custom JSON Schema is valid (Draft 2020-12).")

print("raise_pdf_parsing version:", raise_pdf_parsing.__version__)

## Filtering Smoke Test

This cell builds a small, fully synthetic seven-page annual report in memory -- a research-activities page, a ruled table, a financial-audit page, a photo gallery, a staff directory, an admin-notice page, and a two-column page, all wrapped in a repeated running header/footer -- and runs it through Stage 1 + Stage 2. It is included to verify every filtering category (financial/audit, photo gallery, directory, admin notice, boilerplate) is correctly excluded *before* spending time on a real report, independent of whatever PDF was uploaded above.

In [ ]:
import io

import fitz
from PIL import Image
from raise_pdf_parsing.filter import filter_blocks
from raise_pdf_parsing.parser import parse_pdf

def _square(color, size=220):
    buf = io.BytesIO()
    Image.new("RGB", (size, size), color=color).save(buf, format="PNG")
    return buf.getvalue()

_doc = fitz.open()
_p = _doc.new_page()
_p.insert_text((72, 110), "RESEARCH ACTIVITIES", fontsize=18, fontname="hebo")
_p.insert_text((72, 150), "Funded research continued across departments.", fontsize=11, fontname="tiro")
_p = _doc.new_page()
_p.insert_text((72, 110), "STATEMENT OF ACCOUNTS", fontsize=16, fontname="hebo")
_p.insert_text((72, 150), "Balance Sheet and Auditor's Report attached, trial balance reconciled.", fontsize=11, fontname="tiro")
_p = _doc.new_page()
for _x, _y in [(72, 140), (320, 140), (72, 420), (320, 420)]:
    _p.insert_image(fitz.Rect(_x, _y, _x + 220, _y + 220), stream=_square((200, 200, 200)))
_smoke_pdf = Path("_smoke_test_report.pdf")
_doc.save(_smoke_pdf)
_doc.close()

_smoke_report = parse_pdf(_smoke_pdf)
_smoke_target = filter_blocks(_smoke_report)
_smoke_meta = _smoke_target.filtering_metadata
print(f"Smoke test: {_smoke_meta.output_block_count}/{_smoke_meta.input_block_count} blocks kept, "
      f"{len(_smoke_meta.excluded_pages)} page(s) excluded.")
for _rec in _smoke_meta.excluded_pages:
    print(f"  page {_rec.page_number}: excluded as {_rec.category}")
assert any(r.category == "financial_audit" for r in _smoke_meta.excluded_pages)
assert any(r.category == "photo_gallery" for r in _smoke_meta.excluded_pages)
print("Smoke test passed.")

## Run Stage 1 + Stage 2 on the Uploaded PDF

This cell calls `run_pipeline()` on `input_pdf_filename`. It parses the PDF into layout-aware blocks (Stage 1), then filters out financial/audit pages, photo galleries, staff directories, admin notices, and repeated headers/footers (Stage 2), and writes `report_blocks.json`, `target_blocks.json`, and `filtering_report.json`.

In [ ]:
parsed_path = "report_blocks.json"
target_path = "target_blocks.json"
filtering_report_path = "filtering_report.json"
schema_filter_report_path = "schema_filtering_report.json"

parsed, target = run_pipeline(
    pdf_path=input_pdf_filename,
    parsed_output_path=parsed_path,
    target_output_path=target_path,
    filtering_report_path=filtering_report_path,
)

if custom_schema is not None:
    target, schema_filter_report = filter_target_by_schema(target, custom_schema)
    target_payload = target.model_dump(exclude_none=False)
    TargetBlocksDocument.model_validate(target_payload)
    Path(target_path).write_text(json.dumps(target_payload, indent=2, ensure_ascii=False), encoding="utf-8")
    Path(schema_filter_report_path).write_text(json.dumps(schema_filter_report, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"Custom schema removed {schema_filter_report['removed_block_count']} block(s).")

meta = target.filtering_metadata
print(f"Parsed {parsed.page_count} pages, {len(parsed.blocks)} blocks.")
print(f"Kept {meta.output_block_count} blocks across {len(meta.retained_pages)} pages.")
print(f"Excluded {len(meta.excluded_pages)} page(s).")

## Review the Filtering Decisions

This cell prints which pages were excluded and why, and previews the first few kept blocks of `target_blocks.json`. It is included to verify the filtering result before handing the file to the Stage 3 owner.

In [ ]:
for record in target.filtering_metadata.excluded_pages:
    print(f"page {record.page_number}: {record.category} -- {record.reason}")

print()
print(f"Removed {target.filtering_metadata.removed_boilerplate_block_count} boilerplate block(s).")

print()
print("First 5 kept blocks:")
for block in target.blocks[:5]:
    print(f"  [{block.page_number}] {block.block_type_guess:<10} {block.text[:70]!r}")

## Download the Deliverables

This cell downloads `report_blocks.json` (Stage 1 output), `target_blocks.json` (Stage 2 output -- the file to hand to the Stage 3 owner), and `filtering_report.json` (the audit trail of built-in exclusions). When a custom schema is used, it also downloads `schema_filtering_report.json`.

In [ ]:
files.download(parsed_path)
files.download(target_path)
files.download(filtering_report_path)
if custom_schema is not None:
    files.download(schema_filter_report_path)